# Notebook 03 — Baseline Model

**Project:** Wafer Defect Detection & Yield Risk Dashboard  
**Phase:** Phase 3 — Baseline Model

## Goals

1. Establish a non-deep-learning benchmark before training a CNN
2. Train a Logistic Regression classifier on flattened 64×64 wafer maps
3. Train a Random Forest classifier (on a stratified subset for speed)
4. Evaluate both models with per-class precision, recall, and F1-score
5. Plot confusion matrices
6. Document results in the experiment log

## Why This Step Matters

A baseline model makes the project professional. Without it, there is no way
to know how much the CNN actually improves over simpler approaches.
The baseline also reveals which defect classes are hard to separate
even with a simple model.

**Important:** We use Macro F1 as the primary metric — not accuracy.
With 85.2% of samples being `none`, a model that predicts `none` for every
wafer would reach 85% accuracy but 0% on every defect class.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

from src.preprocessing import load_processed, compute_class_weights
from src.data_loader import DEFECT_CLASSES, LABEL_MAP_INV
from src.evaluate import compute_metrics, print_metrics, plot_confusion_matrix, compare_experiments

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path('../outputs/figures')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup complete.')
print('Defect classes:', DEFECT_CLASSES)

## 2. Load Processed Data

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = load_processed('../data/processed')

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

# Flatten (N, 1, 64, 64) -> (N, 4096) for sklearn
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat   = X_val.reshape(len(X_val), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

print(f'Flattened train shape: {X_train_flat.shape}')

## 3. Experiment 001 — Logistic Regression

Logistic Regression is a strong linear baseline. Fast to train, interpretable,
and a fair test of whether the defect patterns are linearly separable.

We use `class_weight='balanced'` to handle the 989x imbalance automatically.

Expected runtime: **3–8 minutes** on the full training set.

In [ ]:
print('Training Logistic Regression...')
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='saga',       # fastest solver for large datasets
    n_jobs=-1,
    random_state=42,
    C=1.0,
)
lr.fit(X_train_flat, y_train)
print('Training complete.')

joblib.dump(lr, MODELS_DIR / 'baseline_logreg.pkl')
print('Model saved: models/baseline_logreg.pkl')

In [ ]:
y_pred_lr_val  = lr.predict(X_val_flat)
y_pred_lr_test = lr.predict(X_test_flat)

metrics_lr_val  = compute_metrics(y_val,  y_pred_lr_val)
metrics_lr_test = compute_metrics(y_test, y_pred_lr_test)

print_metrics(metrics_lr_val,  'Logistic Regression — Validation Set')
print_metrics(metrics_lr_test, 'Logistic Regression — Test Set')

In [ ]:
plot_confusion_matrix(
    y_test, y_pred_lr_test,
    title='Logistic Regression — Confusion Matrix (Test Set, Normalized)',
    save_path=OUTPUT_DIR / 'confusion_matrix_logreg.png',
    normalize=True,
)
print('Saved: outputs/figures/confusion_matrix_logreg.png')

## 4. Experiment 002 — Random Forest (Stratified Subset)

Random Forest on the full 121k training set would take 30–60 minutes.
We use a stratified 30,000-sample subset — enough to get a fair benchmark
without excessive runtime.

Expected runtime: **5–10 minutes**.

In [ ]:
# Stratified subsample
SUBSET_SIZE = 30_000
rng = np.random.default_rng(42)
subset_idx = []
for cls_idx in np.unique(y_train):
    cls_mask = np.where(y_train == cls_idx)[0]
    n_take = max(1, int(SUBSET_SIZE * len(cls_mask) / len(y_train)))
    chosen = rng.choice(cls_mask, size=min(n_take, len(cls_mask)), replace=False)
    subset_idx.extend(chosen)

subset_idx = np.array(subset_idx)
X_sub = X_train_flat[subset_idx]
y_sub = y_train[subset_idx]

print(f'Subset size: {len(X_sub):,} samples')
print('Class distribution in subset:')
for idx, cnt in zip(*np.unique(y_sub, return_counts=True)):
    print(f'  {LABEL_MAP_INV[idx]:<15} {cnt:>6,}')

In [ ]:
print('Training Random Forest on subset...')
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
    max_depth=None,
    min_samples_leaf=2,
)
rf.fit(X_sub, y_sub)
print('Training complete.')

joblib.dump(rf, MODELS_DIR / 'baseline_rf.pkl')
print('Model saved: models/baseline_rf.pkl')

In [ ]:
y_pred_rf_val  = rf.predict(X_val_flat)
y_pred_rf_test = rf.predict(X_test_flat)

metrics_rf_val  = compute_metrics(y_val,  y_pred_rf_val)
metrics_rf_test = compute_metrics(y_test, y_pred_rf_test)

print_metrics(metrics_rf_val,  'Random Forest — Validation Set')
print_metrics(metrics_rf_test, 'Random Forest — Test Set')

In [ ]:
plot_confusion_matrix(
    y_test, y_pred_rf_test,
    title='Random Forest — Confusion Matrix (Test Set, Normalized)',
    save_path=OUTPUT_DIR / 'confusion_matrix_rf.png',
    normalize=True,
)
print('Saved: outputs/figures/confusion_matrix_rf.png')

## 5. Comparison Table

In [ ]:
compare_experiments({
    'LogReg (full train)':      metrics_lr_test,
    'RandomForest (30k sub)':   metrics_rf_test,
})

## 6. Which Classes Are Hardest?

Look for classes where both models have low F1 — these are the ones the CNN needs to handle better.

In [ ]:
print(f"{'Class':<15} {'LR F1':>10} {'RF F1':>10} {'Support':>10}")
print('-' * 50)
for cls in DEFECT_CLASSES:
    lr_f1 = metrics_lr_test['per_class'][cls]['f1']
    rf_f1 = metrics_rf_test['per_class'][cls]['f1']
    sup   = metrics_lr_test['per_class'][cls]['support']
    flag  = ' <-- hard' if lr_f1 < 0.4 and rf_f1 < 0.4 else ''
    print(f"{cls:<15} {lr_f1:>10.4f} {rf_f1:>10.4f} {sup:>10,}{flag}")

## 7. Summary

Fill in after running:

| Model | Accuracy | Macro F1 | Weighted F1 |
|---|---|---|---|
| Logistic Regression | — | — | — |
| Random Forest (30k) | — | — | — |

**Takeaways:**
- The CNN must beat the baseline Macro F1 to justify the added complexity.
- Classes with low F1 in both baselines will require special attention in the CNN.

**Next step:** `notebooks/04_cnn_training.ipynb`